# Rolling backtest — feature chu kỳ mua

Ta so sánh bộ **18 feature** với bộ **18 + chu kỳ mua** trên các giai đoạn đánh giá nối tiếp nằm hoàn toàn trong train hiện tại. Mỗi mô hình chỉ được học từ cutoff trước giai đoạn đánh giá; một cutoff embargo ngăn cửa sổ nhãn train chạm cutoff đầu tiên của fold đánh giá.

Nhóm chu kỳ đã được chọn từ validation trong thử nghiệm trước. Vì thế các fold dưới đây là bằng chứng bổ sung về độ ổn định, **không phải xác nhận độc lập cuối cùng**. Notebook chỉ đọc tập train; không đọc validation hay test.

## 1. Dữ liệu và chu kỳ tại cutoff

Với `(s-180,s]`, tính khoảng cách giữa các ngày có basket. Nếu không đủ ngày mua, median/std/ratio chưa xác định; chỉ báo cho biết số khoảng cách sẵn có. Logistic fit imputer và scaler chỉ trên train của từng fold; LightGBM xử lý thiếu trực tiếp. `QUANTITY` không được dùng.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss,log_loss,roc_auc_score
ROOT=Path.cwd()
if ROOT.name == 'research' and ROOT.parent.name == 'notebooks': ROOT = ROOT.parent
if ROOT.name=='notebooks':ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
if (ROOT/'.deps').is_dir():sys.path.insert(0,str(ROOT/'.deps'))
from propensity_v2.prepared_data import load_prepared_split
from lightgbm import LGBMClassifier
DATA=ROOT/'artifacts'/'data_preparation';OUT=ROOT/'artifacts'/'research'/'cadence_rolling_backtest'
train,base_features=load_prepared_split(DATA,'train')
tx=pd.read_csv(ROOT/'sample_data'/'dunnhumby'/'transaction_data.csv',usecols=['household_key','DAY'])
active=(tx.loc[tx.DAY<=train.snapshot_day.max(),['household_key','DAY']].drop_duplicates().sort_values(['household_key','DAY']))
cadence_features=['cadence_median_gap_180d','cadence_gap_std_180d','cadence_recency_ratio','cadence_has_gap','cadence_has_two_gaps']
parts=[]
for cutoff in sorted(train.snapshot_day.unique()):
    current=train.loc[train.snapshot_day.eq(cutoff),['household_key','snapshot_day','recency_days']].set_index('household_key').copy()
    days=active.loc[active.DAY.between(cutoff-179,cutoff)].copy()
    days['gap']=days.groupby('household_key').DAY.diff()
    gaps=days.groupby('household_key').gap.agg(['count','median','std'])
    current['cadence_median_gap_180d']=gaps['median']
    current['cadence_gap_std_180d']=gaps['std']
    current['cadence_has_gap']=gaps['count'].ge(1).reindex(current.index,fill_value=False).astype(int)
    current['cadence_has_two_gaps']=gaps['count'].ge(2).reindex(current.index,fill_value=False).astype(int)
    current['cadence_recency_ratio']=current.recency_days/current.cadence_median_gap_180d
    parts.append(current.reset_index()[['household_key','snapshot_day']+cadence_features])
cadence=pd.concat(parts,ignore_index=True)
data=train.merge(cadence,on=['household_key','snapshot_day'],validate='one_to_one')
assert len(data)==len(train) and not data[cadence_features].isin([np.inf,-np.inf]).any().any()
print('Household-snapshot train:',len(data),'| cutoff:',data.snapshot_day.nunique())

Household-snapshot train: 129834 | cutoff: 52


## 2. Kiểm tra phép tính trên các hộ mẫu

Với 100 household-snapshot train, đối chiếu median khoảng cách và điều kiện có đủ dữ liệu trực tiếp từ ngày hoạt động; kiểm tra ranh giới cutoff cho từng fold ở bước tiếp theo.

In [2]:
active_by_household={h:g.DAY.to_numpy() for h,g in active.groupby('household_key',sort=False)}
for r in data.sample(100,random_state=42).itertuples(index=False):
    d=active_by_household[r.household_key]; d=np.sort(d[(d>r.snapshot_day-180)&(d<=r.snapshot_day)])
    gaps=np.diff(d)
    assert r.cadence_has_gap==int(len(gaps)>=1) and r.cadence_has_two_gaps==int(len(gaps)>=2)
    if len(gaps):assert np.isclose(r.cadence_median_gap_180d,np.median(gaps))
    else:assert pd.isna(r.cadence_median_gap_180d)
print('PASS: 100 đối chiếu dữ liệu nguồn.')

PASS: 100 đối chiếu dữ liệu nguồn.


## 3. Thiết kế rolling folds và kịch bản bất thường

Bốn fold đánh giá 6 cutoff liên tiếp tại các giai đoạn khác nhau. Giữa train và fold đánh giá bỏ một cutoff; bảo đảm nhãn train trưởng thành **trước** cutoff đánh giá đầu tiên.

Thực hiện hai kịch bản huấn luyện: giữ toàn bộ, hoặc loại các snapshot có **cửa sổ nhãn 7 ngày hay lịch sử 180 ngày** chứa ngày 278. Với mỗi mô hình đã fit, đánh giá cả trên toàn fold lẫn tập con không có các cửa sổ này. Như vậy có thể tách ảnh hưởng của việc thay đổi train khỏi việc đổi mẫu đánh giá. Vì lọc 180 ngày có thể làm một số fold đánh giá rỗng, bảng ghi rõ số dòng; không gán chỉ số cho fold rỗng.

Chỉ xét ngày 278 trong train; ngày 643 nằm ngoài phạm vi notebook. Cờ chỉ đánh dấu cửa sổ có khả năng chịu ảnh hưởng, không chứng minh lỗi. Lịch sử lifetime như `observed_tenure_days` vẫn có thể mang ảnh hưởng lâu hơn.

In [3]:
snapshots=sorted(data.snapshot_day.unique())
folds=[]
for number,start in enumerate([20,28,36,44],1):
    eval_days=snapshots[start:start+6]; train_days=snapshots[:start-1]; embargo=snapshots[start-1]
    assert len(eval_days)==6 and max(train_days)+7<min(eval_days)
    folds.append({'fold':number,'train_min':min(train_days),'train_max':max(train_days),
                  'embargo':embargo,'eval_min':min(eval_days),'eval_max':max(eval_days),
                  'train_days':train_days,'eval_days':eval_days})
display(pd.DataFrame([{k:v for k,v in fold.items() if k not in ['train_days','eval_days']} for fold in folds]))

,fold,train_min,train_max,embargo,eval_min,eval_max
0,1,180,306,313,320,355
1,2,180,362,369,376,411
2,3,180,418,425,432,467
3,4,180,474,481,488,523


## 4. Huấn luyện và đánh giá cùng cấu hình

Cả hai bộ feature dùng cấu hình giống thử nghiệm trước. Brier, log loss và ROC-AUC tính trên toàn fold; lift top 10% tính riêng mỗi cutoff và trung bình không trọng số. Tại điểm đồng hạng, tính kỳ vọng chọn ngẫu nhiên. Tập con đánh giá sạch được tạo bằng cờ lịch sử 180 ngày và cờ cửa sổ nhãn.

In [4]:
def make_model(name):
    if name=='Logistic Regression':return make_pipeline(SimpleImputer(strategy='median',keep_empty_features=True),StandardScaler(),LogisticRegression(max_iter=2000,random_state=42))
    return LGBMClassifier(n_estimators=200,learning_rate=.05,num_leaves=15,min_child_samples=100,random_state=42,verbosity=-1,n_jobs=2)
def lift(y,p):
    y=np.asarray(y);p=np.asarray(p);k=int(np.ceil(.1*len(y)));t=np.partition(p,len(p)-k)[len(p)-k]
    high=p>t;tied=p==t;precision=(y[high].sum()+(k-high.sum())*y[tied].mean())/k
    return precision/y.mean() if y.mean() else np.nan
def evaluate(y,p,cutoff):
    frame=pd.DataFrame({'y':np.asarray(y),'p':np.asarray(p),'cutoff':np.asarray(cutoff)})
    weekly=[lift(g.y,g.p) for _,g in frame.groupby('cutoff')]
    return {'brier':brier_score_loss(y,p),'log_loss':log_loss(y,p,labels=[0,1]),
            'roc_auc':roc_auc_score(y,p) if pd.Series(y).nunique()==2 else np.nan,
            'weekly_lift':float(np.nanmean(weekly)) if np.isfinite(weekly).any() else np.nan}
rows=[]
for fold in folds:
    fit_all=data.loc[data.snapshot_day.isin(fold['train_days'])]
    eval_all=data.loc[data.snapshot_day.isin(fold['eval_days'])]
    for train_scenario in ['toàn bộ','loại nhãn và lịch sử 180d']:
        fit=fit_all if train_scenario=='toàn bộ' else fit_all.loc[~(fit_all.label_window_touches_anomaly|fit_all.history_180d_touches_anomaly)]
        assert len(fit)>0 and fit.target_any_basket.nunique()==2
        for model_name in ['Logistic Regression','LightGBM']:
            for feature_set,columns in [('18 feature',base_features),('18 + chu kỳ',base_features+cadence_features)]:
                model=make_model(model_name);model.fit(fit[columns],fit.target_any_basket)
                predictions=model.predict_proba(eval_all[columns])[:,1]
                for eval_scenario in ['toàn bộ','loại nhãn và lịch sử 180d']:
                    mask=np.ones(len(eval_all),dtype=bool) if eval_scenario=='toàn bộ' else ~(eval_all.label_window_touches_anomaly|eval_all.history_180d_touches_anomaly).to_numpy()
                    record={'fold':fold['fold'],'model':model_name,'feature_set':feature_set,
                            'train_scenario':train_scenario,'eval_scenario':eval_scenario,
                            'train_rows':len(fit),'eval_rows':int(mask.sum()),
                            'eval_cutoffs':eval_all.loc[mask,'snapshot_day'].nunique()}
                    if mask.any():record.update(evaluate(eval_all.loc[mask,'target_any_basket'],predictions[mask],eval_all.loc[mask,'snapshot_day']))
                    else:record.update({m:np.nan for m in ['brier','log_loss','roc_auc','weekly_lift']})
                    rows.append(record)
results=pd.DataFrame(rows)
display(results.head(12))

,fold,model,feature_set,train_scenario,eval_scenario,train_rows,eval_rows,eval_cutoffs,brier,log_loss,roc_auc,weekly_lift
0,1,Logistic Regression,18 feature,toàn bộ,toàn bộ,47418,14982,6,0.161429,0.488817,0.843539,1.851210
1,1,Logistic Regression,18 feature,toàn bộ,loại nhãn và lịch sử 180d,47418,0,0,NaN,NaN,NaN,NaN
2,1,Logistic Regression,18 + chu kỳ,toàn bộ,toàn bộ,47418,14982,6,0.160516,0.486419,0.845025,1.855057
3,1,Logistic Regression,18 + chu kỳ,toàn bộ,loại nhãn và lịch sử 180d,47418,0,0,NaN,NaN,NaN,NaN
4,1,LightGBM,18 feature,toàn bộ,toàn bộ,47418,14982,6,0.159548,0.483447,0.846735,1.853675
5,1,LightGBM,18 feature,toàn bộ,loại nhãn và lịch sử 180d,47418,0,0,NaN,NaN,NaN,NaN
6,1,LightGBM,18 + chu kỳ,toàn bộ,toàn bộ,47418,14982,6,0.158734,0.481327,0.848245,1.858799
7,1,LightGBM,18 + chu kỳ,toàn bộ,loại nhãn và lịch sử 180d,47418,0,0,NaN,NaN,NaN,NaN
8,1,Logistic Regression,18 feature,loại nhãn và lịch sử 180d,toàn bộ,32437,14982,6,0.161649,0.489560,0.843175,1.852504
9,1,Logistic Regression,18 feature,loại nhãn và lịch sử 180d,loại nhãn và lịch sử 180d,32437,0,0,NaN,NaN,NaN,NaN


## 5. So sánh feature và độ nhạy

Mỗi chênh lệch là **(18 + chu kỳ) − (18 feature)**, cùng model, fold và kịch bản. Brier/log loss âm và ROC-AUC/lift dương là cải thiện. Các fold phụ thuộc nhau và độ dài train tăng dần; không xem bốn fold là bằng chứng thống kê độc lập.

In [5]:
keys=['fold','model','train_scenario','eval_scenario']
reference=results.loc[results.feature_set.eq('18 feature'),keys+['brier','log_loss','roc_auc','weekly_lift']]
reference=reference.rename(columns={m:'base_'+m for m in ['brier','log_loss','roc_auc','weekly_lift']})
comparison=results.loc[results.feature_set.eq('18 + chu kỳ')].merge(reference,on=keys,validate='one_to_one')
for metric in ['brier','log_loss','roc_auc','weekly_lift']:comparison['delta_'+metric]=comparison[metric]-comparison['base_'+metric]
display(comparison[['fold','model','train_scenario','eval_scenario','train_rows','eval_rows','eval_cutoffs',
                    'delta_brier','delta_log_loss','delta_roc_auc','delta_weekly_lift']])
summary=(comparison.groupby(['model','train_scenario','eval_scenario'],as_index=False)
         .agg(folds_available=('delta_brier','count'),mean_delta_brier=('delta_brier','mean'),
              mean_delta_log_loss=('delta_log_loss','mean'),mean_delta_auc=('delta_roc_auc','mean'),
              mean_delta_lift=('delta_weekly_lift','mean')))
display(summary)

,fold,model,train_scenario,eval_scenario,train_rows,eval_rows,eval_cutoffs,delta_brier,delta_log_loss,delta_roc_auc,delta_weekly_lift
0,1,Logistic Regression,toàn bộ,toàn bộ,47418,14982,6,-0.000913,-0.002397,0.001486,0.003847
1,1,Logistic Regression,toàn bộ,loại nhãn và lịch sử 180d,47418,0,0,NaN,NaN,NaN,NaN
2,1,LightGBM,toàn bộ,toàn bộ,47418,14982,6,-0.000814,-0.002120,0.001509,0.005124
3,1,LightGBM,toàn bộ,loại nhãn và lịch sử 180d,47418,0,0,NaN,NaN,NaN,NaN
4,1,Logistic Regression,loại nhãn và lịch sử 180d,toàn bộ,32437,14982,6,-0.000962,-0.002617,0.001535,0.005132
5,1,Logistic Regression,loại nhãn và lịch sử 180d,loại nhãn và lịch sử 180d,32437,0,0,NaN,NaN,NaN,NaN
6,1,LightGBM,loại nhãn và lịch sử 180d,toàn bộ,32437,14982,6,-0.001003,-0.002817,0.001944,0.005151
7,1,LightGBM,loại nhãn và lịch sử 180d,loại nhãn và lịch sử 180d,32437,0,0,NaN,NaN,NaN,NaN
8,2,Logistic Regression,toàn bộ,toàn bộ,67394,14982,6,-0.000631,-0.001968,0.000882,-0.000047
9,2,Logistic Regression,toàn bộ,loại nhãn và lịch sử 180d,67394,0,0,NaN,NaN,NaN,NaN


,model,train_scenario,eval_scenario,folds_available,mean_delta_brier,mean_delta_log_loss,mean_delta_auc,mean_delta_lift
0,LightGBM,loại nhãn và lịch sử 180d,loại nhãn và lịch sử 180d,2,-0.000736,-0.001810,0.001398,-0.002512
1,LightGBM,loại nhãn và lịch sử 180d,toàn bộ,4,-0.000722,-0.001971,0.001396,0.004096
2,LightGBM,toàn bộ,loại nhãn và lịch sử 180d,2,-0.000459,-0.001342,0.001040,-0.000759
3,LightGBM,toàn bộ,toàn bộ,4,-0.000621,-0.001573,0.001234,-0.000005
4,Logistic Regression,loại nhãn và lịch sử 180d,loại nhãn và lịch sử 180d,2,-0.001151,-0.003826,0.001366,-0.001182
5,Logistic Regression,loại nhãn và lịch sử 180d,toàn bộ,4,-0.001005,-0.003033,0.001332,0.001597
6,Logistic Regression,toàn bộ,loại nhãn và lịch sử 180d,2,-0.000768,-0.002841,0.001307,0.001839
7,Logistic Regression,toàn bộ,toàn bộ,4,-0.000795,-0.002486,0.001228,0.001870


**Kết quả:** Với train và đánh giá đầy đủ, thêm chu kỳ làm Brier và log loss giảm ở cả 4 fold cho cả Logistic Regression và LightGBM. ROC-AUC tăng ở cả 4 fold; lift trung bình không tăng nhất quán, đặc biệt LightGBM giảm trong fold 4. Khi loại cửa sổ nhãn và lịch sử 180 ngày ở phía đánh giá, fold 1 và 2 rỗng; chỉ fold 3 (2 cutoff) và fold 4 (6 cutoff) còn dữ liệu. Trên các fold sạch còn lại, Brier và log loss vẫn giảm trong cả hai kịch bản train, nhưng mẫu đánh giá đã thay đổi và lifetime vẫn có thể mang ảnh hưởng.

Đây là bằng chứng ủng hộ chu kỳ mua như một ứng viên cải thiện xác suất; chưa chứng minh cải thiện lift ổn định hay độc lập với dữ liệu đã dùng để lựa chọn feature.


## 6. Kết luận và bàn giao

Đọc kết quả cùng số fold và quy mô còn lại sau lọc. Các fold sạch có thể chỉ đại diện những cutoff muộn; không so sánh trung bình khác tập mẫu như thể chỉ do loại train. Test và validation không được đọc trong notebook. Việc chọn feature dựa trên validation trước đó vẫn phải được nêu trong báo cáo.

In [6]:
OUT.mkdir(parents=True,exist_ok=True)
results.to_csv(OUT/'fold_metrics.csv',index=False,encoding='utf-8-sig')
comparison.to_csv(OUT/'fold_feature_differences.csv',index=False,encoding='utf-8-sig')
summary.to_csv(OUT/'summary.csv',index=False,encoding='utf-8-sig')
print('Đã lưu:',OUT)

Đã lưu: artifacts\cadence_rolling_backtest
